In [1]:
import pandas as pd

In [5]:
df = pd.read_csv('labels.csv')
for row in df.rolling(1):
    print(row['audio_path'][0])

300_AUDIO.wav


In [ ]:
str(df['participant_id'])}_P', str(path['audio_path'])

# Data cleaning

### first we clean audio files to remove background noises and enhance speaker voices

we can test to see if this makes things better

In [ ]:
import os
import subprocess

INPUT_FILE = r"/Users/gurusai/Desktop/DAIC_Raw/300_P/300_AUDIO.wav"
OUTPUT_DIR = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Step 1: Demucs
print("Running Demucs...")
subprocess.run([
    "demucs",
    "--two-stems=vocals",
    INPUT_FILE
])

base = os.path.splitext(os.path.basename(INPUT_FILE))[0]
demucs_out = f"separated/htdemucs/{base}/vocals.wav"

# Step 2: FFmpeg cleaning
output_path = os.path.join(OUTPUT_DIR, "cleanedv2.wav")

print("Applying FFmpeg filters...")

subprocess.run([
    "ffmpeg",
    "-i", demucs_out,
    "-af",
    # Chain of filters:
    # 1. highpass → remove low rumble
    # 2. lowpass → remove harsh high freq noise
    # 3. afftdn → noise reduction
    # 4. compand → smooth dynamics (makes speech clearer)
    "highpass=f=80, lowpass=f=12000, afftdn=nf=-22:tn=1, equalizer=f=3000:t=q:w=1:g=3, compand",
    output_path,
    "-y"
])

print(f"Done! Output saved at: {output_path}")

# second we get rid of the interviewer segments of the convo and encode them separately

In [ ]:
import pandas as pd
import os

def _normalize_visual_df(df, prefix):
    # Some OpenFace exports use a leading space in " timestamp".
    df = df.rename(columns={" timestamp": "timestamp"}).copy()

    required_keys = ["frame", "timestamp"]
    missing = [c for c in required_keys if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns {missing} in {prefix} file")

    # Prefix all non-join columns to avoid collisions (e.g., confidence/success).
    feature_cols = [c for c in df.columns if c not in required_keys]
    rename_map = {c: f"{prefix}_{c.strip()}" for c in feature_cols}
    return df.rename(columns=rename_map)

def process_full_multimodal_alignment(p_id, p_folder):
    # 1. Load transcript
    df_t = pd.read_csv(os.path.join(p_folder, f"{p_id}_TRANSCRIPT.csv"), sep='\t')

    # 2. Load OpenFace streams
    df_clnf = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_features3D.txt"))
    df_gaze = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_gaze.txt"))
    df_pose = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_pose.txt"))
    df_au = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_AUs.txt"))

    # 3. Normalize and disambiguate columns before merging
    df_clnf = _normalize_visual_df(df_clnf, "clnf")
    df_gaze = _normalize_visual_df(df_gaze, "gaze")
    df_pose = _normalize_visual_df(df_pose, "pose")
    df_au = _normalize_visual_df(df_au, "au")

    # Merge visual features on frame+timestamp into one master table.
    df_visual = (
        df_clnf
        .merge(df_gaze, on=["frame", "timestamp"], how="inner")
        .merge(df_pose, on=["frame", "timestamp"], how="inner")
        .merge(df_au, on=["frame", "timestamp"], how="inner")
    )

    processed_turns = []
    last_question = "Initial Greeting"

    # Handle slight transcript schema variations.
    stop_col = "stop_time" if "stop_time" in df_t.columns else "end_time"

    for _, row in df_t.iterrows():
        text = str(row["value"]).lower()

        # Interviewer turn updates context anchor.
        if row["speaker"] == "Ellie":
            if "?" in text or any(q in text for q in ["how", "why", "tell me", "describe", "do you", "did you", "can you","when","have you"]): # we can work on this later to be more robust, but for now this captures most questions.
                if len(text.split()) > 3:
                    last_question = text
            continue

        # Participant turn gets aligned to visual window.
        if row["speaker"] == "Participant":
            start, stop = row["start_time"], row[stop_col]
            visual_mask = (df_visual["timestamp"] >= start) & (df_visual["timestamp"] <= stop)
            turn_visual_data = df_visual[visual_mask]

            if not turn_visual_data.empty:
                processed_turns.append({
                    "participant_id": p_id,
                    "question_context": last_question,
                    "answer_text": text,
                    "start_time": start,
                    "stop_time": stop,
                    "visual_features": turn_visual_data.drop(columns=["frame", "timestamp"]).values,
                })

    return processed_turns

# Example usage:
p_id = "301"
p_folder = r"/Users/gurusai/Desktop/DAIC_Raw/301_P"
aligned_data = process_full_multimodal_alignment(p_id, p_folder)
print(f"Aligned participant turns: {len(aligned_data)}")

Aligned participant turns: 104


In [13]:
aligned_data

[{'participant_id': '301',
  'question_context': 'Initial Greeting',
  'answer_text': 'thank you',
  'start_time': 32.738,
  'stop_time': 33.068,
  'visual_features': array([[  0.989402,   1.      , -47.9324  , ...,   0.      ,   0.      ,
            1.      ],
         [  0.988848,   1.      , -47.8198  , ...,   0.      ,   0.      ,
            0.      ],
         [  0.989969,   1.      , -47.709   , ...,   0.      ,   0.      ,
            0.      ],
         ...,
         [  0.99106 ,   1.      , -48.7375  , ...,   0.      ,   0.      ,
            0.      ],
         [  0.990795,   1.      , -49.0733  , ...,   0.      ,   0.      ,
            0.      ],
         [  0.990567,   1.      , -49.4904  , ...,   0.      ,   0.      ,
            0.      ]], shape=(10, 250))},
 {'participant_id': '301',
  'question_context': 'Initial Greeting',
  'answer_text': 'mmm k',
  'start_time': 42.088,
  'stop_time': 42.518,
  'visual_features': array([[  0.989053,   1.      , -46.6997  , ...,  